<a href="https://colab.research.google.com/github/achyutarag/b2b_solutions/blob/main/Solution_3_and_6_B2B_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# B2B AI Reliability Toolkit

Two standalone, shippable dev tools for taming enterprise RAG failures.

- **Part 1 —  Zero-Bloat Contextual Grounding Kit** *(Solution 3)*
  Catches and auto-patches RAG retrieval failures at the database layer instead of bloating the prompt.
- **Part 2 —  Golden Test Set Builder** *(Solution 6)*
  Builds bias-free, stratified evaluation sets so you know exactly how your AI performs across every customer segment.

Each part is self-contained — Part 2 does not depend on any state from Part 1.


---
# 🧠 Part 1 — Zero-Bloat Contextual Grounding Kit
### Contextual Grounding for Enterprise RAG

**The B2B Pain Point:** When an enterprise RAG system returns "I don't know" or hallucinates, engineers usually try to fix it by blindly writing massive system prompts or chunking data into huge text blocks. This introduces noise and causes the LLM to lose information.

**The Solution:** A developer tool that isolates exactly *why* a query failed. Instead of letting an LLM guess over thousands of tokens of context, this tool catches **empty retrievals** or **low-relevancy scores** at the database level and automatically triggers a localized fix — a lightweight query rewrite or a metadata filter — so the retrieval database does the heavy lifting and the prompt stays clean.

**The Pitch:** *"Stop fixing your data problems with longer LLM prompts. We pinpoint and patch RAG retrieval failures at the database source for 100% predictable grounding."*

**Tools used:** SQL (SQLite) for failure auditing · ChromaDB (vector DB) for retrieval · A local T5 transformer for lightweight query rewriting


## How It Routes — Three Conditions

**1. The ACCURATE path**
*Question: "What time does the IT helpdesk close?"*
The database matches this strongly against the IT data chunk. The score flies past 0.70, bypassing the triage fixes entirely and going straight to the main LLM.

**2. The LOW RELEVANCY path**
*Question: "Can I get a refund on my vacation days?"*
The database gets confused — it has a chunk about "refunds" and a chunk about "vacation days (PTO)," but they don't cleanly belong together. The score lands in the grey zone (0.30–0.70), triggering an autonomous query rewrite plus an active-status metadata filter.

**3. The EMPTY RETRIEVAL path**
*Question: "How do I fix a spaceship engine?"*
The vector database finds zero correlation with company documents. The score drops below 0.30, the query is flagged as an empty retrieval, blocked from hitting the primary LLM, and routed to a keyword-expansion tool to find a better search term.


In [ ]:
# ------------------------------------------------------------------
# 1.1  SETUP — Install dependencies
# ------------------------------------------------------------------
!pip install chromadb transformers


In [ ]:
# ------------------------------------------------------------------
# 1.2  LOCAL LLM ENGINE — Lightweight T5 query rewriter
# ------------------------------------------------------------------
# Loaded via the native tokenizer/model classes (rather than the high-level
# pipeline() API) to keep full control over generation args, and suppress
# the noisy transformers/weights warnings that clutter notebook output.

import warnings
warnings.filterwarnings("ignore")

from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

def run_local_t5(prompt_text):
    """Generate a short completion locally with flan-t5-small (no API cost)."""
    inputs = tokenizer(prompt_text, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# ------------------------------------------------------------------
# 1.3  VECTOR DATABASE — Sample company knowledge base
# ------------------------------------------------------------------
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="company_knowledge",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    collection.add(
        documents=[
            "Our company refund policy allows returns within 30 days with a receipt.",
            "The office IT helpdesk is open from 9 AM to 5 PM, Monday through Friday.",
            "Employees receive 15 days of paid time off (PTO) per calendar year."
        ],
        metadatas=[{"status": "active"}, {"status": "active"}, {"status": "active"}],
        ids=["policy_001", "it_001", "hr_001"]
    )


In [ ]:
# ------------------------------------------------------------------
# 1.4  SQL AUDIT LOG — Persist every triage failure
# ------------------------------------------------------------------
import sqlite3
from datetime import datetime

def init_audit_log():
    """Create the rag_triage_logs table if it doesn't already exist."""
    conn = sqlite3.connect("rag_analytics.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS rag_triage_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT,
            original_query TEXT,
            failure_condition TEXT,
            intercepted_score REAL,
            applied_fix_action TEXT,
            patched_query TEXT,
            resolved_context TEXT
        )
    """)
    conn.commit()
    conn.close()

def log_triage_event(original_query, condition, score, action, patched_query, final_context):
    """Insert one triage event into the audit log."""
    conn = sqlite3.connect("rag_analytics.db")
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO rag_triage_logs
        (timestamp, original_query, failure_condition, intercepted_score, applied_fix_action, patched_query, resolved_context)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        original_query,
        condition,
        score,
        action,
        patched_query,
        final_context[:150] + "..." if final_context else "None"
    ))
    conn.commit()
    conn.close()
    print("📊 [Telemetry Engine] Failure intercepted and securely logged to SQL database.")

init_audit_log()


In [ ]:
# ------------------------------------------------------------------
# 1.5  DYNAMIC SCORE CALIBRATION — The "Universal Score"
# ------------------------------------------------------------------
# Raw cosine-distance scores are model-specific. Every score gets rescaled
# onto a shared 0.0-1.0 "universal" metric so the routing logic stays
# portable across different embedding models (see 1.8 for a live proof).

MODEL_CEILING = 1.0  # true mathematical identity ceiling

# Probe with a nonsense numeric string to establish a realistic semantic floor
floor_results = collection.query(query_texts=["987321456000000"], n_results=1)
MODEL_FLOOR = 1.0 - min(floor_results["distances"][0][0], 1.0)

if MODEL_CEILING <= MODEL_FLOOR:  # safety check
    MODEL_FLOOR = 0.0


In [ ]:
# ------------------------------------------------------------------
# 1.6  MEMORY-DRIVEN PATCH ENGINE — Few-shot query rewriting
# ------------------------------------------------------------------
# Looks up the most recent successful fixes for this failure type and feeds
# them to the local T5 model as few-shot examples, so the system gets
# smarter every time it logs a new fix.

def generate_generalized_patch(user_query, failure_condition):
    conn = sqlite3.connect("rag_analytics.db")
    cursor = conn.cursor()
    cursor.execute("""
        SELECT original_query, patched_query
        FROM rag_triage_logs
        WHERE failure_condition = ?
        ORDER BY id DESC LIMIT 3
    """, (failure_condition,))
    past_examples = cursor.fetchall()
    conn.close()

    if not past_examples:  # cold start — no history yet, fall back to a seed example
        if failure_condition == "LOW_RELEVANCY":
            past_examples = [("helpdesk open?", "IT helpdesk schedule")]
        else:
            past_examples = [("spaceship fix", "company equipment maintenance policy")]

    prompt = f"Context: You are an expert search optimizer correcting a {failure_condition} error.\n"
    prompt += "Study these past successful corrections:\n\n"
    for old_query, fixed_query in past_examples:
        prompt += f"Input: {old_query} -> Output: {fixed_query}\n"
    prompt += f"\nNow optimize this new query:\nInput: {user_query} -> Output:"

    return run_local_t5(prompt)


In [ ]:
# ------------------------------------------------------------------
# 1.7  RUNTIME PIPELINE — Live triage on a real query
# ------------------------------------------------------------------
print("\n--- RUNTIME WITH DYNAMIC STATE GENERALIZATION ---")
user_query = input("Ask a question: ")

db_results = collection.query(query_texts=[user_query], n_results=1)
raw_distance = db_results["distances"][0][0]
raw_similarity = 1.0 - min(raw_distance, 1.0)

universal_score = (raw_similarity - MODEL_FLOOR) / (MODEL_CEILING - MODEL_FLOOR)
universal_score = max(0.0, min(universal_score, 1.0))
print(f"📊 Universal Score for this query: {universal_score:.4f}")

if universal_score < 0.40:
    print("🚨 Condition Caught: EMPTY_RETRIEVAL")
    patched_query = generate_generalized_patch(user_query, "EMPTY_RETRIEVAL")
    print(f"🔄 Smart Patch Deployed via Memory: '{patched_query}'")

    final_results = collection.query(query_texts=[patched_query], n_results=1)
    final_context = final_results["documents"][0][0] if final_results["documents"] else "None"
    log_triage_event(user_query, "EMPTY_RETRIEVAL", universal_score,
                      "GENERALIZED_KEYWORD_EXPANSION", patched_query, final_context)

elif universal_score < 0.85:
    print("⚠️ Condition Caught: LOW_RELEVANCY")
    patched_query = generate_generalized_patch(user_query, "LOW_RELEVANCY")
    print(f"🔧 Smart Patch Deployed via Memory: '{patched_query}'")

    final_results = collection.query(query_texts=[patched_query], where={"status": "active"}, n_results=1)
    final_context = final_results["documents"][0][0] if final_results["documents"] else "None"
    log_triage_event(user_query, "LOW_RELEVANCY", universal_score,
                      "GENERALIZED_REWRITE_AND_FILTER", patched_query, final_context)

else:
    print("✅ Condition Caught: ACCURATE. Passing directly to primary LLM.")
    final_context = db_results["documents"][0][0]


> **Note on thresholds:** this runtime demo uses fixed gates (`0.40` / `0.85`) so it's easy to read top-to-bottom.
> Part 2 of this notebook (the Golden Test Set Builder) shows how to replace fixed guesses like this with
> data-driven, risk-weighted gates computed from real evaluation data.


In [ ]:
# ------------------------------------------------------------------
# 1.8  MULTI-MODEL CONSISTENCY TEST — Universal Score in action
# ------------------------------------------------------------------
# Demonstrates that the universal-score abstraction produces the SAME
# routing decision regardless of which embedding model's raw score range
# we feed it — proving the triage logic is model-agnostic.

model_profiles = {
    "Model_A_HF_MiniLM (Tight Range)": {"floor": 0.05, "ceiling": 0.50},
    "Model_B_OpenAI_Text3 (Broad Range)": {"floor": 0.35, "ceiling": 0.95}
}

print("\n🚀 --- STARTING MULTI-MODEL CONSISTENCY TEST ---")
user_query = input("Ask a test question (e.g., 'what time does the IT helpdesk close?'): ")

db_results = collection.query(query_texts=[user_query], n_results=1)
raw_distance = db_results["distances"][0][0]
base_score = 1.0 - min(raw_distance, 1.0)

for model_name, bounds in model_profiles.items():
    print(f"\nEvaluating pipeline behavior using: **{model_name}**")

    # Simulate how this model would express our base score in its own coordinate space
    simulated_raw_score = bounds["floor"] + (base_score * (bounds["ceiling"] - bounds["floor"]))
    print(f"   ↳ [Raw Model Vector Space Output]: {simulated_raw_score:.2f}")

    # Strip the model-specific bias back out to the shared 0.0-1.0 universal metric
    universal_score = (simulated_raw_score - bounds["floor"]) / (bounds["ceiling"] - bounds["floor"])
    universal_score = max(0.0, min(universal_score, 1.0))
    print(f"   ↳ 🚀 [Universal Metric Conversion]: **{universal_score:.2f}**")

    if universal_score < 0.30:
        print("   ↳ 🚨 [Interception Triggered]: EMPTY_RETRIEVAL ➔ Routing to Keyword Expansion.")
    elif universal_score < 0.70:
        print("   ↳ ⚠️ [Interception Triggered]: LOW_RELEVANCY ➔ Routing to Query Rewrite & Filter.")
    else:
        print("   ↳ ✅ [Pass Through Granted]: ACCURATE ➔ Routing straight to Main LLM.")


In [ ]:
# ------------------------------------------------------------------
# 1.9  EVALUATION HARNESS — Ground truth ranking
# ------------------------------------------------------------------
# Re-connects to the persistent collection independently, so this section
# can be run on its own without re-running 1.1-1.8 first.

import pandas as pd

chroma_client = chromadb.Client()
collection = chroma_client.get_collection(name="company_knowledge")

evaluation_set = [
    {"query": "What is the policy for product returns?", "expected_bucket": "ACCURATE"},
    {"query": "when is the IT helpdesk open until on Fridays?", "expected_bucket": "ACCURATE"},
    {"query": "How many days of PTO do employees get?", "expected_bucket": "ACCURATE"},
    {"query": "Can I get a refund on my vacation days?", "expected_bucket": "LOW_RELEVANCY"},
    {"query": "Helpdesk open for return policy?", "expected_bucket": "LOW_RELEVANCY"},
    {"query": "What is the corporate policy on flying spaceships?", "expected_bucket": "EMPTY_RETRIEVAL"}
]

results_log = []
for item in evaluation_set:
    user_query = item["query"]
    db_results = collection.query(query_texts=[user_query], n_results=1)
    raw_distance = db_results["distances"][0][0]
    raw_similarity = 1.0 - min(raw_distance, 1.0)

    universal_score = (raw_similarity - MODEL_FLOOR) / (MODEL_CEILING - MODEL_FLOOR)
    universal_score = max(0.0, min(universal_score, 1.0))

    results_log.append({
        "Query": user_query,
        "Expected Bucket": item["expected_bucket"],
        "Universal Score": round(universal_score, 4)
    })

df = pd.DataFrame(results_log).sort_values(by="Universal Score", ascending=False)
print("\n📊 ─── GROUND TRUTH EVALUATION DISTRIBUTION ───")
print(df.to_string(index=False))


In [ ]:
# ------------------------------------------------------------------
# 1.10  SYSTEM MEMORY DIAGNOSTIC — Audit the SQL log
# ------------------------------------------------------------------
import sqlite3

def check_system_memory():
    conn = sqlite3.connect("rag_analytics.db")
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT COUNT(*) FROM rag_triage_logs")
        total_logs = cursor.fetchone()[0]

        cursor.execute("""
            SELECT id, timestamp, original_query, failure_condition, applied_fix_action, patched_query
            FROM rag_triage_logs
            ORDER BY id DESC LIMIT 5
        """)
        recent_records = cursor.fetchall()

        print("📊 [System Memory Diagnostic]")
        print(f"Total historical failures intercepted and fixed: {total_logs}\n")

        if total_logs == 0:
            print("⚪ The SQL database is currently empty. No failures have been logged yet.")
        else:
            print("👇 Recent Autonomous Fixes Saved in Memory:")
            for row in recent_records:
                print(f"  • [ID {row[0]}] {row[1]} | State: {row[3]}")
                print(f"    Raw Input: '{row[2]}'")
                print(f"    Deployed Fix ({row[4]}): '{row[5]}'\n")
    except sqlite3.OperationalError:
        print("❌ The table 'rag_triage_logs' does not exist yet. Run cell 1.4 first.")
    finally:
        conn.close()

check_system_memory()


In [ ]:
# ------------------------------------------------------------------
# 1.11  MAINTENANCE UTILITY — Purge a bad telemetry record (optional)
# ------------------------------------------------------------------
# Handy during development if a bad/test row pollutes the few-shot prompt.
# Not run automatically — call wipe_bad_telemetry(record_id) manually.

import sqlite3

def wipe_bad_telemetry(record_id):
    conn = sqlite3.connect("rag_analytics.db")
    cursor = conn.cursor()
    cursor.execute("DELETE FROM rag_triage_logs WHERE id = ?", (record_id,))
    conn.commit()
    conn.close()
    print(f"🧹 Record {record_id} successfully purged from the memory engine.")

# Example: wipe_bad_telemetry(1)


---
# 📊 Part 2 — Golden Test Set Builder
### Stratified Data Splitting for Unbiased AI Evaluation

**The B2B Pain Point:** A B2B enterprise builds an AI customer support agent and tests it on a random sample of past support tickets. It scores beautifully — then hallucinates the moment a customer asks a complex enterprise billing question, because complex billing questions were only 1% of the database and got missed entirely by random sampling. The evaluation was biased and over-represented simple password-reset queries.

**The Solution:** A data auditing and stratification tool. It ingests historical operational data, tags metadata (user type, query complexity, account tier), and automatically extracts a perfectly balanced, multi-variable stratified "Golden Test Set" — including rare, high-risk singleton cases that random sampling would otherwise drop.

**The Pitch:** *"If your AI test data is unbalanced, your AI will fail on your most valuable clients. We audit your operational data and compile perfectly balanced benchmarking datasets so you know exactly how your AI behaves across every customer segment."*

**Tools used:** Pandas / NumPy for categorical filtering and stratified sampling · Scikit-learn for the split engine and evaluation metrics


In [ ]:
# ------------------------------------------------------------------
# 2.1  SETUP
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
# ------------------------------------------------------------------
# 2.2  COMPILE OPERATIONAL LOG DATA — 60+ simulated tickets
# ------------------------------------------------------------------
# Six core operational archetypes spanning different business tiers and
# complexities, expanded into a noisy, realistic 60+ row warehouse, plus
# one rare, high-risk singleton.

templated_queries = [
    {"query": "Reset password link broken", "tier": "Freemium", "complexity": "Simple"},
    {"query": "How to update billing credit card", "tier": "Tier-1 Enterprise", "complexity": "Simple"},
    {"query": "API returning 500 internal error on custom endpoint", "tier": "Tier-1 Enterprise", "complexity": "Complex"},
    {"query": "SLA compliance report download", "tier": "Tier-1 Enterprise", "complexity": "Complex"},
    {"query": "Where is the cancel button", "tier": "Freemium", "complexity": "Simple"},
    {"query": "Can I get a refund on accidental renewal", "tier": "Freemium", "complexity": "Complex"},
]

expanded_logs = []
for i in range(10):
    for idx, item in enumerate(templated_queries):
        base_score = 0.85 if item["complexity"] == "Simple" else 0.48
        simulated_score = round(base_score + np.random.uniform(-0.15, 0.12), 4)

        expanded_logs.append({
            "query": f"{item['query']} [Session_ID_{i*10+idx}]",
            "tier": item["tier"],
            "complexity": item["complexity"],
            "Universal Score": simulated_score,
            "Expected Bucket": "ACCURATE" if item["complexity"] == "Simple" else "LOW_RELEVANCY"
        })

# Inject one rare, high business-risk singleton that a naive random sample would miss
expanded_logs.append({
    "query": "LEGAL NOTICE: Immediate termination of master service agreement [CRITICAL_SINGLETON]",
    "tier": "Strategic_Enterprise",
    "complexity": "Critical_Legal",
    "Universal Score": 0.2105,
    "Expected Bucket": "LOW_RELEVANCY"
})

df_large_raw = pd.DataFrame(expanded_logs)
df_large_raw["stratify_key"] = df_large_raw["tier"] + "_" + df_large_raw["complexity"]


In [ ]:
# ------------------------------------------------------------------
# 2.3  SINGLETON-SAFE STRATIFIED SAMPLING
# ------------------------------------------------------------------
# sklearn's train_test_split throws an error on any stratify class with a
# single member. We isolate those high-risk singletons first, run the
# stratified split on the remaining "splittable" rows, then add the
# singletons back in — guaranteeing rare classes are never dropped.

class_counts = df_large_raw["stratify_key"].value_counts()
singleton_keys = class_counts[class_counts == 1].index

df_singletons = df_large_raw[df_large_raw["stratify_key"].isin(singleton_keys)]
df_splittable = df_large_raw[~df_large_raw["stratify_key"].isin(singleton_keys)]
print("⚙️ [Middleware Engine]: Isolated singletons to protect long-tail edge cases.")

golden_dense_slice, _ = train_test_split(
    df_splittable,
    train_size=20,
    stratify=df_splittable["stratify_key"],
    random_state=42
)

final_large_golden_set = pd.concat([df_singletons, golden_dense_slice], axis=0)


In [ ]:
# ------------------------------------------------------------------
# 2.4  BIAS-DETECTION AUDIT REPORT
# ------------------------------------------------------------------
print("\n📊 ─── LARGE-SCALE BIAS-DETECTION AUDIT REPORT ───")
print(f"• Raw Operational Log Inventory: {len(df_large_raw)} messy queries processed.")
print(f"• Golden Test Set Footprint: {len(final_large_golden_set)} benchmark queries extracted.")
print("─" * 65)
print("Input Warehouse Strata Proportions:")
print(df_large_raw["stratify_key"].value_counts(normalize=True).round(2).to_string())
print("─" * 65)
print("Balanced Golden Test Set Strata Proportions:")
print(final_large_golden_set["stratify_key"].value_counts(normalize=True).round(2).to_string())
print("─" * 65)

print("\n📋 PREVIEW OF BALANCED REVENUE-CRITICAL BENCHMARK DATASET:")
print(final_large_golden_set[["query", "tier", "complexity"]].head(8).to_string(index=False))


## Objective

This module maps continuous vector-embedding confidence metrics (the "Universal Score") into explicit, hard operational routing boundaries. Using the balanced, multi-variable Golden Test Set generated above, the engine automatically determines the optimal threshold gates needed to protect production LLMs from toxic, low-relevancy, or out-of-bounds queries.

## Technical Methodology

This module moves the evaluation pipeline from a static heuristic check to a dynamic, risk-weighted optimization engine:

1. **Semantic Gap Analysis** — profiles the validation data into statistical boundaries (ACCURATE floor vs. LOW_RELEVANCY max).
2. **Dynamic Strategy Routing** — when a clear semantic gap exists, applies a cost-weighted optimization (default cost ratio 4:1), biasing the threshold toward precision since a false positive (a hallucination reaching an enterprise client) is weighted as 4x worse than a false negative.
3. **Statistical Friction Handling** — when real-world embedding scores overlap between classes, the engine drops the standard midpoint and uses a 90th-percentile Low-Relevancy filter instead, pushing the safety gate to the edge of the noisy data cloud for maximum precision protection.
4. **Categorical Evaluation** — scores are binned into `EMPTY_RETRIEVAL`, `LOW_RELEVANCY`, and `ACCURATE`, then scored with a standard scikit-learn confusion matrix and classification report.


In [ ]:
# ------------------------------------------------------------------
# 2.5  AUTO-CALIBRATION ENGINE — Risk-weighted threshold gates
# ------------------------------------------------------------------
def diagnose_and_calibrate_large_scale(df, business_strategy="cost", cost_ratio=4.0):
    """
    Diagnoses semantic gaps and auto-calibrates thresholds based on a chosen
    business risk strategy. Hardened to handle large stratified datasets that
    contain score overlaps between classes.
    """
    print(f"\n🩺 ─── AUTO-DIAGNOSTIC CALIBRATION ENGINE ({business_strategy.upper()} STRATEGY) ───")

    try:
        accurate_min = df[df["Expected Bucket"] == "ACCURATE"]["Universal Score"].min()
        low_rel_max = df[df["Expected Bucket"] == "LOW_RELEVANCY"]["Universal Score"].max()
        low_rel_min = df[df["Expected Bucket"] == "LOW_RELEVANCY"]["Universal Score"].min()
    except Exception:
        print("❌ Error: Evaluation dataset structure mismatch. Verify Expected Bucket values.")
        return None, None

    overlap_detected = low_rel_max >= accurate_min
    recommended_empty_gate = round(low_rel_min * 0.5, 4)

    if not overlap_detected:
        gap = accurate_min - low_rel_max
        if business_strategy == "precision":
            recommended_low_rel_gate = round(low_rel_max + (gap * 0.75), 4)
            print("🎯 Strategy Applied: Clean separation. PRECISION BIAS pushed threshold UP.")
        elif business_strategy == "recall":
            recommended_low_rel_gate = round(low_rel_max + (gap * 0.25), 4)
            print("🎣 Strategy Applied: Clean separation. RECALL BIAS dropped threshold DOWN.")
        elif business_strategy == "cost":
            weight = cost_ratio / (1.0 + cost_ratio)
            recommended_low_rel_gate = round(low_rel_max + (gap * weight), 4)
            print(f"💰 Strategy Applied: Clean separation. Optimized with a {cost_ratio}:1 risk ratio.")
        else:
            recommended_low_rel_gate = round((low_rel_max + accurate_min) / 2, 4)
            print("⚖️ Strategy Applied: Clean separation. Split the gap evenly.")
    else:
        print("⚠️ STATISTICAL FRICTION DETECTED: Overlap zone present between Low Relevancy and Accurate queries.")
        if business_strategy == "cost" and cost_ratio > 1.0:
            recommended_low_rel_gate = round(
                np.percentile(df[df["Expected Bucket"] == "LOW_RELEVANCY"]["Universal Score"], 90), 4
            )
            print("🎯 Strategy Applied: PRECISION PROTECTION. Raised gate to 90th percentile of Low Relevancy cloud.")
        else:
            recommended_low_rel_gate = round((low_rel_max + accurate_min) / 2, 4)
            print("⚖️ Strategy Applied: Overlap resolved using simple center midpoint.")

    print("\n✅ Analysis Complete. Large-Scale Boundaries Mapping:")
    print(f"   • ACCURATE Floor Score:    {accurate_min}")
    print(f"   • LOW_RELEVANCY Max Score: {low_rel_max}")
    print(f"   • Calibrated Gates:        EMPTY < {recommended_empty_gate} ↔ LOW_REL < {recommended_low_rel_gate}")

    return recommended_empty_gate, recommended_low_rel_gate


# Run the calibration using a 4:1 business penalty cost ratio
df_eval = final_large_golden_set.copy()
empty_gate, low_rel_gate = diagnose_and_calibrate_large_scale(df_eval, business_strategy="cost", cost_ratio=4.0)

# Other risk strategies available for different B2B use cases:
# diagnose_and_calibrate_large_scale(df_eval, business_strategy="precision")  # high-stakes financial/medical triage
# diagnose_and_calibrate_large_scale(df_eval, business_strategy="recall")     # e-commerce discovery triage


In [ ]:
# ------------------------------------------------------------------
# 2.6  CONFUSION MATRIX & CLASSIFICATION REPORT — Golden Set evaluation
# ------------------------------------------------------------------
df_eval["Predicted Bucket"] = df_eval["Universal Score"].apply(
    lambda x: "EMPTY_RETRIEVAL" if x < empty_gate else ("LOW_RELEVANCY" if x < low_rel_gate else "ACCURATE")
)

labels = ["ACCURATE", "LOW_RELEVANCY"]
y_true = df_eval["Expected Bucket"]
y_pred = df_eval["Predicted Bucket"]

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])

print("\n📊 ─── LARGE-SCALE CONFUSION MATRIX & EVALUATION REPORT ───")
print("Confusion Matrix Grid:")
print("─" * 60)
print(cm_df)
print("─" * 60)

print("\nDetailed Performance Metrics Against Balanced Golden Suite:")
print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
